In [15]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px

from dash import Dash, dcc, html, Input, Output


In [16]:
# Load the dashboard data
dashboard_df = pd.read_csv("MIG_dashboard_inventory_forecast.csv")

dashboard_df.head()


,site_id,date,ensemble,sarimax,xgboost,random_forest,opening_inventory_sim,closing_inventory_sim,silo_capacity,deliveries_tonnes,reorder_alert,stockout_risk
0,SITE_001,2025-01-01,53.266630,58.846242,52.647118,48.30653,5.85,0.0,448,0.0,True,True
1,SITE_001,2025-01-02,53.482827,59.494834,52.647118,48.30653,0.00,0.0,448,0.0,True,True
2,SITE_001,2025-01-03,53.504280,59.559194,52.647118,48.30653,0.00,0.0,448,0.0,True,True
3,SITE_001,2025-01-04,53.506409,59.565581,52.647118,48.30653,0.00,0.0,448,0.0,True,True
4,SITE_001,2025-01-05,53.506621,59.566214,52.647118,48.30653,0.00,0.0,448,0.0,True,True


In [17]:
dashboard_df['date'] = pd.to_datetime(dashboard_df['date'])


In [18]:
# Prepare site list.
sites = sorted(dashboard_df['site_id'].unique())
sites[:10]


['SITE_001',
 'SITE_002',
 'SITE_003',
 'SITE_004',
 'SITE_005',
 'SITE_006',
 'SITE_007',
 'SITE_008',
 'SITE_009',
 'SITE_010']

In [19]:
# Create helper function for charts. 
# Your presentation colour palette
maroon = "#800000"
gold = "#C9A86A"
navy = "#0A1A44"
teal = "#1B7F7A"
grey = "#4A4A4A"
light_grey = "#D9D9D9"

def forecast_chart(df_site):
    fig = go.Figure()

    # Ensemble Forecast (Primary line)
    fig.add_trace(go.Scatter(
        x=df_site['date'],
        y=df_site['ensemble'],
        mode='lines',
        name='Ensemble Forecast',
        line=dict(color=maroon, width=4)
    ))

    # SARIMAX
    fig.add_trace(go.Scatter(
        x=df_site['date'],
        y=df_site['sarimax'],
        mode='lines',
        name='SARIMAX',
        line=dict(color=gold, width=2.5, dash='dot')
    ))

    # XGBoost
    fig.add_trace(go.Scatter(
        x=df_site['date'],
        y=df_site['xgboost'],
        mode='lines',
        name='XGBoost',
        line=dict(color=navy, width=2.5, dash='dash')
    ))

    # Random Forest
    fig.add_trace(go.Scatter(
        x=df_site['date'],
        y=df_site['random_forest'],
        mode='lines',
        name='Random Forest',
        line=dict(color=teal, width=2.5, dash='dashdot')
    ))

    fig.update_layout(
        title="8‑Week Cement Consumption Forecast",
        xaxis_title="Date",
        yaxis_title="Tonnes",
        template="plotly_white",
        font=dict(size=14, color=grey),
        legend=dict(
            bgcolor="rgba(0,0,0,0)",
            bordercolor=light_grey,
            borderwidth=1
        )
    )

    return fig


In [20]:
# 4.2 Inventory projectio chart. 
# Your presentation colour palette
maroon = "#800000"
gold = "#C9A86A"
navy = "#0A1A44"
teal = "#1B7F7A"
grey = "#4A4A4A"
light_grey = "#D9D9D9"

def inventory_chart(df_site):
    fig = go.Figure()

    # Closing Inventory (main operational line)
    fig.add_trace(go.Scatter(
        x=df_site['date'],
        y=df_site['closing_inventory_sim'],
        mode='lines',
        name='Closing Inventory',
        line=dict(color=navy, width=4)
    ))

    # Silo Capacity (reference line)
    fig.add_trace(go.Scatter(
        x=df_site['date'],
        y=df_site['silo_capacity'],
        mode='lines',
        name='Silo Capacity',
        line=dict(color=maroon, width=3, dash='dot')
    ))

    fig.update_layout(
        title="Projected Inventory Levels",
        xaxis_title="Date",
        yaxis_title="Tonnes",
        template="plotly_white",
        font=dict(size=14, color=grey),
        legend=dict(
            bgcolor="rgba(0,0,0,0)",
            bordercolor=light_grey,
            borderwidth=1
        )
    )

    return fig


In [21]:
# 4.3 Reorder alert table
def reorder_table(df_site):
    alerts = df_site[df_site['reorder_alert'] == True][['date','closing_inventory_sim']]
    return alerts


In [23]:
# 5 Build the Dash App 
app = Dash(__name__)

app.layout = html.Div([
    html.H1("MIG Cement Forecasting Dashboard", style={'textAlign': 'center'}),

    html.Label("Select Site:"),
    dcc.Dropdown(
        id='site_selector',
        options=[{'label': s, 'value': s} for s in sites],
        value=sites[0],
        clearable=False
    ),

    dcc.Graph(id='forecast_graph'),
    dcc.Graph(id='inventory_graph'),

    html.H3("Reorder Alerts"),
    html.Div(id='alert_table')
])


In [24]:
# callbacks to update the graphs and table based on selected site

@app.callback(
    [Output('forecast_graph', 'figure'),
     Output('inventory_graph', 'figure'),
     Output('alert_table', 'children')],
    [Input('site_selector', 'value')]
)
def update_dashboard(site):
    df_site = dashboard_df[dashboard_df['site_id'] == site]

    fig_forecast = forecast_chart(df_site)
    fig_inventory = inventory_chart(df_site)

    alerts = reorder_table(df_site)

    if alerts.empty:
        alert_html = html.P("No reorder alerts.")
    else:
        alert_html = html.Table([
            html.Thead(html.Tr([html.Th("Date"), html.Th("Closing Inventory")])),

            html.Tbody([
                html.Tr([
                    html.Td(row['date']),
                    html.Td(f"{row['closing_inventory_sim']:.2f}")
                ]) for _, row in alerts.iterrows()
            ])
        ])

    return fig_forecast, fig_inventory, alert_html


In [25]:
# 7. Run the dashboard.  
if __name__ == "__main__":
    app.run(debug=True)
